# 09 — Playback response: when to play a call, and which one

After a vocalization is scored, the site does not always play a sound back. This notebook is that decision.

The input is the same three scores the report already shows:

- **Valence** — unpleasant to pleasant, about −5 to +5
- **Arousal** — how stirred up the bird is, about 0 to 5
- **Social engagement** — pulled away from others to actively seeking contact, about −5 to +5

Those scores are first read as a **combined state** (for example “Happy and Socially Engaged” or “Serene and Restful”). The playback model then maps that state, together with the raw scores, onto one of five actions:

| Mode | What the owner is offered |
| --- | --- |
| **echo** | A recording that made other lovebirds answer |
| **cheer** | A recording that made other lovebirds turn and listen |
| **companion** | The same kind of friendly recording, offered as optional company |
| **rest** | No audio. The bird is too low on energy |
| **quiet** | No audio. The bird is already worked up, and another call would pile on |

This is a priority list, not a newly trained classifier. The same rules run on the site in `apps/Chirp/src/lib/playbackResponse.ts`. Rest is checked first, then quiet, then cheer, then echo. Anything left is companion.


In [1]:
from pathlib import Path

import pandas as pd

CHIRP = Path(r"D:\CHIRP")
CLIP_MEANS = CHIRP / "analysis" / "outputs" / "behavior_clip_means.csv"


## 1. Combined state

The report names one state. Rules are tried from top to bottom, and the first match wins. The playback step uses that name, so the list below is part of the model even though it is shared with the report.

Scores in the examples later use the same axes as the site: valence and social engagement can be negative; arousal is a non-negative energy score.


In [2]:
STATE_RULES = [
    ("Distressed and Avoidant", lambda v, a, s: v < -2.5 and s < -2),
    ("Stressed and Defensive", lambda v, a, s: v < -1 and a >= 3 and s < 0),
    ("Irritable and Agitated", lambda v, a, s: v < -0.5 and a >= 3.5),
    ("Uneasy and On Guard", lambda v, a, s: v < -0.5 and a >= 2.5 and s < 1.5),
    ("Wary but Coping", lambda v, a, s: v < 0 and v >= -1.5 and a < 3.5 and s >= -1),
    ("Withdrawn and Uncomfortable", lambda v, a, s: v < -1 and a < 2 and s < -0.5),
    ("Lonely and Subdued", lambda v, a, s: v < 0.5 and s < -1 and a < 2.8),
    ("Subdued and Low Spirit", lambda v, a, s: v < 0.5 and a < 2 and abs(s) < 1),
    ("Euphoric and Outgoing", lambda v, a, s: v >= 3.5 and a >= 3.5 and s >= 3),
    ("Playful and Enthusiastic", lambda v, a, s: v >= 2.8 and a >= 3.2 and s >= 2.5),
    ("Vocal and Attention-Seeking", lambda v, a, s: v >= 1.5 and a >= 3.5 and s >= 3),
    ("Happy and Socially Engaged", lambda v, a, s: v >= 2.8 and a >= 2.5 and s >= 2.8),
    ("Cheerful and Conversational", lambda v, a, s: v >= 2.2 and a >= 2 and a < 3.5 and s >= 2 and s < 3.5),
    ("Upbeat and Interactive", lambda v, a, s: v >= 1.8 and a >= 2.5 and s >= 1.5 and s < 2.8),
    ("Content and Companionable", lambda v, a, s: v >= 1.5 and a < 2.6 and s >= 2),
    ("Pleased and Attentive", lambda v, a, s: v >= 1 and v < 2.5 and a >= 2 and a < 3.2 and s >= 0.5 and s < 2.5),
    ("Bright and Attentive", lambda v, a, s: v >= 0.3 and v < 2.2 and a >= 2.2 and s < 2),
    ("Relaxed and Content", lambda v, a, s: v >= 1 and a < 2 and s >= 0 and s < 2.5),
    ("Serene and Restful", lambda v, a, s: v >= -0.3 and a < 1.4 and s >= -0.5),
    ("Content but Low Energy", lambda v, a, s: v >= 1 and a < 2 and s < 1),
    ("Calm and Observant", lambda v, a, s: v >= -0.5 and v < 1.8 and a >= 1.4 and a < 2.8 and s >= -0.5 and s < 2),
    ("Quietly Curious", lambda v, a, s: v >= 0 and v < 2 and a >= 1.4 and a < 2.5 and s >= 0.5 and s < 2),
    ("Mellow and Self-Sufficient", lambda v, a, s: v >= 0.5 and a < 2.2 and s < 1),
    ("Energetic but Self-Contained", lambda v, a, s: v >= 0.5 and a >= 3 and s < 1.5),
    ("Excited and Independent", lambda v, a, s: v >= 0 and a >= 3.2 and s < 0.5),
    ("Alert but Reserved", lambda v, a, s: a >= 2.5 and s < 0.5 and v >= -0.5 and v < 2),
    ("Restless and Unsettled", lambda v, a, s: a >= 3.5 and v >= -0.5 and v < 1.5 and s < 1.5),
    ("Neutral and Even-Tempered", lambda v, a, s: abs(v) < 1 and a >= 1.5 and a < 2.8 and abs(s) < 1.5),
    ("Balanced and Steady", lambda v, a, s: True),
]


def combined_state(valence, arousal, social):
    for name, rule in STATE_RULES:
        if rule(valence, arousal, social):
            return name
    return "Balanced and Steady"


## 2. The playback decision

Five actions, checked in this order. The first one that matches is the one the page uses.

1. **Rest.** The named state is “Serene and Restful” or “Content but Low Energy”, or arousal is below 0.8 (the “Very Calm” band). A low mood with arousal below 1.4 is also rest: a sad bird that is nearly out of energy is not offered a cheering call.
2. **Quiet.** The state is “Stressed and Defensive”, “Irritable and Agitated”, or “Uneasy and On Guard”. The same action is used when valence is below −0.5 and arousal is at least 3, even if the name landed elsewhere. Another bird call is withheld.
3. **Cheer.** The state is one of the low-mood names below, or valence itself is negative. The page offers a warm, happy recording.
4. **Echo.** The state is socially engaged (the names below), or social engagement is at least 1.8 and valence is not negative. The page offers a call that answers back.
5. **Companion.** Everything else. A friendly recording is optional, not something the bird needs.

Low-mood names: Withdrawn and Uncomfortable, Distressed and Avoidant, Lonely and Subdued, Subdued and Low Spirit, Wary but Coping.

Echo names: Euphoric and Outgoing, Playful and Enthusiastic, Happy and Socially Engaged, Cheerful and Conversational, Upbeat and Interactive, Vocal and Attention-Seeking, Content and Companionable.


In [3]:
REST_STATES = {"Serene and Restful", "Content but Low Energy"}
QUIET_STATES = {"Stressed and Defensive", "Irritable and Agitated", "Uneasy and On Guard"}
CHEER_STATES = {
    "Withdrawn and Uncomfortable",
    "Distressed and Avoidant",
    "Lonely and Subdued",
    "Subdued and Low Spirit",
    "Wary but Coping",
}
ECHO_STATES = {
    "Euphoric and Outgoing",
    "Playful and Enthusiastic",
    "Happy and Socially Engaged",
    "Cheerful and Conversational",
    "Upbeat and Interactive",
    "Vocal and Attention-Seeking",
    "Content and Companionable",
}


def playback_mode(valence, arousal, social):
    """Return rest, quiet, cheer, echo, or companion."""
    state = combined_state(valence, arousal, social)
    low_mood = state in CHEER_STATES or valence < 0

    if state in REST_STATES or arousal < 0.8 or (low_mood and arousal < 1.4):
        return "rest", state
    if state in QUIET_STATES or (valence < -0.5 and arousal >= 3):
        return "quiet", state
    if low_mood:
        return "cheer", state
    if state in ECHO_STATES or (social >= 1.8 and valence >= 0):
        return "echo", state
    return "companion", state


def plays_audio(mode):
    return mode in {"echo", "cheer", "companion"}


## 3. Worked examples

Each row is a score triple the report might show. The last column is what the page does with it.


In [4]:
examples = pd.DataFrame(
    [
        ("settled and very quiet", 0.5, 1.0, 0.0),
        ("content but drained", 1.2, 1.6, -0.2),
        ("happy, almost still", 2.0, 0.5, 2.0),
        ("sad and nearly asleep", -1.2, 1.0, -0.5),
        ("wary, still awake", -1.2, 2.2, -0.2),
        ("lonely", 0.2, 2.2, -1.5),
        ("stressed", -1.5, 3.2, -0.5),
        ("agitated", -0.8, 3.6, 0.2),
        ("interactive", 2.2, 2.8, 2.0),
        ("happy and social", 3.0, 2.8, 3.0),
        ("calm and watching", 0.2, 2.0, 0.2),
        ("pleased and social", 1.2, 2.4, 2.2),
    ],
    columns=["case", "valence", "arousal", "social"],
)

resolved = examples.apply(
    lambda row: playback_mode(row.valence, row.arousal, row.social), axis=1, result_type="expand"
)
resolved.columns = ["mode", "state"]
examples = pd.concat([examples, resolved], axis=1)
examples["plays_audio"] = examples["mode"].map(plays_audio)
examples


,case,valence,arousal,social,mode,state,plays_audio
0,settled and very quiet,0.5,1.0,0.0,rest,Serene and Restful,False
1,content but drained,1.2,1.6,-0.2,rest,Content but Low Energy,False
2,"happy, almost still",2.0,0.5,2.0,rest,Content and Companionable,False
3,sad and nearly asleep,-1.2,1.0,-0.5,rest,Wary but Coping,False
4,"wary, still awake",-1.2,2.2,-0.2,cheer,Wary but Coping,True
5,lonely,0.2,2.2,-1.5,cheer,Lonely and Subdued,True
6,stressed,-1.5,3.2,-0.5,quiet,Stressed and Defensive,False
7,agitated,-0.8,3.6,0.2,quiet,Irritable and Agitated,False
8,interactive,2.2,2.8,2.0,echo,Cheerful and Conversational,True
9,happy and social,3.0,2.8,3.0,echo,Happy and Socially Engaged,True


## 4. Which recordings are offered

The pool is **not** chosen from words in the filename (`happy`, `callforresponse`, and so on). It is chosen from what the three listeners did after the sound, in `data/playback_data_combined.xlsx`.

Each listener has a short behavior note plus scores for movement (MI), body posture (BP), and directionality (D), all on 0–5. Listeners who ignored, slept, or froze sit near zero on movement and direction. Listeners who answered sit higher on direction (they turned toward the sound).

A note is read as:

- **reply** — respond, approach, or call back
- **attend** — attention, observe, curious, excited, or interest, and not a reply
- **shut down** — ignore, sleepy, freeze, fluff, or turn away
- **alarm** — alert, fight, startle, or biting

**Echo.** At least two listeners replied, none shut down or alarmed, and mean directionality is at least 2. Rank by `2 × replies + mean directionality − within-listener SD`. Keep at most two clips from the same bird, then take eight.

**Cheer.** Not an echo clip: at most one reply, at least two listeners attended, none shut down or alarmed, mean directionality at least 1.3, mean body posture at least 2. Same bird cap, then eight. Companion uses this pool too.

Rest and quiet never open either pool.


In [5]:
import re

playback = pd.read_excel(CHIRP / "data" / "playback_data_combined.xlsx", sheet_name=0)
means = pd.read_csv(CLIP_MEANS)
means["within_sd"] = means[["MI_within_sd", "BP_within_sd", "D_within_sd"]].mean(axis=1)

REPLY = re.compile(r"respond|response|approach|\bcall\b")
SHUT = re.compile(r"ignore|sleepy|freeze|fluff|turn away")
ALARM = re.compile(r"\balert\b|fight|startl|taken back|biting")
EXCITED = re.compile(r"excit|curious|interest")
ATTEND = re.compile(r"attention|observe|curious|excit|interest|\blook\b|tilt|\bturn\b")


def norm_behavior(value):
    if pd.isna(value):
        return ""
    text = str(value).lower()
    return text.replace("rspond", "respond").replace("appraoch", "approach").replace("respnd", "respond")


def reaction(raw):
    text = norm_behavior(raw)
    if not text.strip():
        return "missing"
    if REPLY.search(text):
        return "reply"
    if ALARM.search(text):
        return "alarm"
    if SHUT.search(text):
        return "shut"
    if ATTEND.search(text):
        return "attend"
    return "other"


rows = []
for _, row in playback.iterrows():
    notes = [norm_behavior(row[f"Receiver{k}Behavior"]) for k in (1, 2, 3)]
    kinds = [reaction(note) for note in notes]
    rows.append({
        "FileName": row["FileName"],
        "reply": kinds.count("reply"),
        "attend": kinds.count("attend"),
        "shut": kinds.count("shut"),
        "alarm": kinds.count("alarm"),
        "excited": int(any(EXCITED.search(note) for note in notes)),
    })

clips = means.merge(pd.DataFrame(rows), on="FileName")
wavs = {path.stem for path in (CHIRP / "data" / "wav_files").glob("*.wav")}
clips = clips[clips["FileName"].isin(wavs)].copy()


def take(frame, limit=8, per_bird=2):
    picked, counts = [], {}
    for _, row in frame.iterrows():
        bird = str(row["FileName"]).split("_")[0]
        if counts.get(bird, 0) >= per_bird:
            continue
        picked.append(row)
        counts[bird] = counts.get(bird, 0) + 1
        if len(picked) == limit:
            break
    return pd.DataFrame(picked)


echo = clips[(clips.reply >= 2) & (clips.alarm == 0) & (clips.shut == 0) & (clips.D_mean >= 2)].copy()
echo["score"] = echo.reply * 2 + echo.D_mean - echo.within_sd
echo_pool = take(echo.sort_values("score", ascending=False))
echo_pool.insert(0, "pool", "echo")

cheer = clips[
    (clips.reply <= 1) & (clips.attend >= 2) & (clips.alarm == 0) & (clips.shut == 0)
    & (clips.D_mean >= 1.3) & (clips.BP_mean >= 2)
].copy()
cheer["score"] = cheer.attend + cheer.excited + 0.4 * cheer.D_mean + 0.2 * cheer.BP_mean - cheer.within_sd
cheer_pool = take(cheer.sort_values("score", ascending=False))
cheer_pool.insert(0, "pool", "cheer")

library = pd.concat([echo_pool, cheer_pool], ignore_index=True)
library[["pool", "FileName", "reply", "attend", "D_mean", "BP_mean", "within_sd", "score"]].round(2)


,pool,FileName,reply,attend,D_mean,BP_mean,within_sd,score
0,echo,Cleo_alarmresponding_1848,3,0,3.33,3.00,0.38,8.95
1,echo,Aster_callingfizzy_1336,3,0,3.67,3.33,0.77,8.90
2,echo,Bobby_readytofly_0905,3,0,3.00,3.00,0.19,8.81
3,echo,Eko_landingcallmate_1343,3,0,3.00,2.67,0.19,8.81
4,echo,Cleo_readytofly_1831,3,0,3.00,3.00,0.53,8.47
5,echo,Aster_warnmate_1326,3,0,3.33,2.00,1.10,8.23
6,echo,Fizzy_respondtomate_1402,3,0,2.33,3.00,0.58,7.76
7,echo,Bobby_callmate_1857,3,0,2.00,2.33,0.53,7.47
8,cheer,Bobby_callmate_1314,0,3,3.00,3.33,0.38,4.48
9,cheer,Bobby_callmate_1326,1,2,3.00,3.00,0.53,4.27


## 5. How a new score is handled

Given `(valence, arousal, social)`:

1. Name the combined state with the ordered rules in section 1.
2. Apply the priority list in section 2 to get `rest`, `quiet`, `cheer`, `echo`, or `companion`.
3. If the mode is `rest` or `quiet`, stop. The page explains why no sound is played.
4. If the mode is `echo`, pick one file from the echo pool. Otherwise pick one from the cheer pool.

The cell below is that lookup for a single triple. Change the three numbers to see which action and which pool come back.


In [6]:
valence, arousal, social = 2.2, 2.8, 2.0
mode, state = playback_mode(valence, arousal, social)
pool = None if not plays_audio(mode) else ("echo" if mode == "echo" else "cheer")

print(f"state: {state}")
print(f"mode:  {mode}")
print(f"pool:  {pool or 'none — do not play audio'}")


state: Cheerful and Conversational
mode:  echo
pool:  echo
